# YOLO Pose Fine-Tuning (small quality set / supplementary to base yolo-pose)

Trains `yolo11x-pose.pt` on the **local NDJSON** quality set (`datasets/auto-label-v1_53.ndjson`, 52 train / 13 val frames) and writes the result to `src/models/pose/yolo11x-pose-trained.pt` so `hoid_model_run.ipynb` picks it up.

Recipe: ***frozen*** backbone (`freeze=10`) so the base detector's proven person/pose features are preserved, and only the pose/detect head adapts to this scene. `optimizer=SGD` keeps the real LR at `lr0=0.001` (the `optimizer=auto` AdamW bump to ~0.002 is what collapsed this set before).


In [ ]:
### Imports & Training Configuration
import shutil
import os
from pathlib import Path

from ultralytics import YOLO

import nest_asyncio2
nest_asyncio2.apply()  # lets ultralytics' async ul:// dataset loader run inside Jupyter's event loop

MODEL_BASE = "models/pose/yolo11x-pose.pt"
DATA_URI   = "datasets/auto-label-v1.ndjson"

EPOCHS        = 30
LR0           = 0.001
OPTIMIZER     = "SGD"        ### keep real LR at lr0=0.001; optimizer=auto would bump to AdamW ~0.002 (collapsed this set before)
FREEZE        = 10           ### freeze backbone: keep base person detection, adapt only the pose/detect head (supplementary to base yolo-pose)
PATIENCE      = 15
IMGSZ         = 640
BATCH         = 4            ### OOM ladder: 4 -> 2 -> switch MODEL_BASE to yolo11l-pose.pt with BATCH=8
DEVICE        = 0

### Scene-specialized augmentation (single fixed CCTV camera)
WARMUP_EPOCHS = 3            ### ramp up so the larger LR does not blast the tiny set on epoch 1
MOSAIC        = 0.0
MIXUP         = 0.0
COPY_PASTE    = 0.0
ERASING       = 0.0
SCALE         = 0.1
FLIPLR        = 0.5          ### relaxed: people cross both sides of frame
HSV_H         = 0.02
HSV_S         = 0.25
HSV_V         = 0.15

PROJECT       = "thesis"
NAME          = "pose"

OUT_WEIGHTS   = Path("models/pose/yolo11x-pose-trained.pt")


In [ ]:
### Freeze BN running stats on FROZEN layers (collapse guard)

def _bn_stabilize(trainer):
    seq = trainer.model.model
    n = len(seq)
    for i in range(n):
        seq[i].train(i >= FREEZE)   # layers < FREEZE -> eval (BN frozen); rest -> train

def _register_bn_stabilize(model):
    model.add_callback("on_train_batch_start", _bn_stabilize)


In [3]:
### Clean the small NDJSON before training: zero out-of-bounds / negative keypoints (idempotent)
import json as _json
from pathlib import Path
from ultralytics.utils import checks as _checks

_uri = Path(_checks.check_file(DATA_URI))
_n_oob = _n_neg = 0
_out = []
for _ln in _uri.read_text(encoding="utf-8").splitlines():
    _ln = _ln.strip()
    if not _ln:
        continue
    _o = _json.loads(_ln)
    if _o.get("type") == "image":
        for _p in (_o.get("annotations") or {}).get("pose") or []:
            _nk = (len(_p) - 5) // 3
            for _k in range(_nk):
                _x = _p[5 + _k * 3]; _y = _p[6 + _k * 3]
                if _x < 0.0 or _x > 1.01:
                    _p[5 + _k * 3] = 0.0; _n_oob += 1
                if _y < 0.0 or _y > 1.01:
                    _p[6 + _k * 3] = 0.0; _n_neg += 1
    _out.append(_ln if _o.get("type") != "image" else _json.dumps(_o, ensure_ascii=False, separators=(",", ":")))
if _n_oob or _n_neg:
    _uri.write_text("\n".join(_out) + "\n", encoding="utf-8")
print(f"data cleanup {DATA_URI}: zeroed {_n_oob} OOB + {_n_neg} negative keypoints")


data cleanup datasets/auto-label-v1.ndjson: zeroed 10 OOB + 17 negative keypoints


In [4]:
### Train
model = YOLO(MODEL_BASE)
_register_bn_stabilize(model)

model.train(
    data         = DATA_URI,
    epochs       = EPOCHS,
    imgsz        = IMGSZ,
    batch        = BATCH,
    freeze       = FREEZE,
    lr0          = LR0,
    warmup_epochs= WARMUP_EPOCHS,
    patience     = PATIENCE,
    device       = DEVICE,
    project      = PROJECT,
    name         = NAME,
    optimizer    = OPTIMIZER,
    exist_ok     = True,
    mosaic       = MOSAIC,
    mixup        = MIXUP,
    copy_paste   = COPY_PASTE,
    erasing      = ERASING,
    scale        = SCALE,
    fliplr       = FLIPLR,
    hsv_h        = HSV_H,
    hsv_s        = HSV_S,
    hsv_v        = HSV_V,
)

save_dir = Path(model.trainer.save_dir)
print("\nrun directory:", save_dir)

### Write the trained pose model where hoid_model_run.ipynb expects it (overwrite)
OUT_WEIGHTS.parent.mkdir(parents=True, exist_ok=True)

shutil.copy(save_dir / "weights" / "best.pt", OUT_WEIGHTS)
print("trained model ->", OUT_WEIGHTS)


New https://pypi.org/project/ultralytics/8.4.139 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.127  Python-3.12.1 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=datasets/auto-label-v1.ndjson, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.0, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.02, hsv_s=0.25, hsv_v=0.15, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, 